In [1]:
!pip install lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=041b6c46774623b7df84ae3223c697db36e4f931ee69dbc9b3dfe28b32d2ef36
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


# **Importing Libraries & Dataset**

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

csv_url = 'https://data.cityofnewyork.us/api/views/fi59-268w/rows.csv?accessType=DOWNLOAD'

try:
    df = pd.read_csv(csv_url)
except Exception as e:
    print(f"Error loading CSV from URL: {e}")

df.head()

,ccpversion,maprojid,magencyacro,magency,magencyname,description,projectid,mindate,maxdate,typecategory,...,commit_total,spent_ccnonexempt,spent_ccexempt,spent_citycost,spent_nccstate,spent_nccfederal,spent_nccother,spent_noncitycost,spent_total,spent_total_checkbooknyc
0,fisa_2026,850SE849,DDC,850,Department of Design and Construction,"Dist WM SE Rplmt in Southeast Clason Point, BX",SE849,10/01/2025,06/01/2032,NaN,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
1,fisa_2026,125HAM14CRSR,DFTA,125,Department for the Aging,CARVER HOUSES SENIOR CENTER COMPUTER LAB,HAM14CRSR,06/01/2027,06/01/2027,Fixed Asset,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
2,fisa_2026,035LRCE12DOT,NYRL,35,New York Research Libraries,SASB:Improvements to the NYPL Vicinity,LRCE12DOT,06/01/2028,06/01/2031,NaN,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
3,fisa_2026,035L21FREEZE,NYRL,35,New York Research Libraries,NYPL Research Libraries - Blast Freezer,L21FREEZE,06/01/2026,06/01/2026,NaN,...,6600.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
4,fisa_2026,850SEK002383,DDC,850,Department of Design and Construction,Combined SE installation & WM replacement in D...,SEK002383,03/15/2023,06/01/2028,Lump Sum,...,13133548.54,138249.54,10298251.65,10436501.19,0.0,0.0,1501431.87,1501431.87,11937933.06,6896227.51


# Data Inspection & Exploration

In [3]:
print('Number of rows and columns in the data set:', df.shape)

# information of column names, data types and memory usage
df.info()

Number of rows and columns in the data set: (11493, 51)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11493 entries, 0 to 11492
Data columns (total 51 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ccpversion                 11493 non-null  object 
 1   maprojid                   11493 non-null  object 
 2   magencyacro                11493 non-null  object 
 3   magency                    11493 non-null  int64  
 4   magencyname                11493 non-null  object 
 5   description                11493 non-null  object 
 6   projectid                  11493 non-null  object 
 7   mindate                    11493 non-null  object 
 8   maxdate                    11493 non-null  object 
 9   typecategory               10607 non-null  object 
 10  plannedcommit_ccnonexempt  11493 non-null  int64  
 11  plannedcommit_ccexempt     11493 non-null  int64  
 12  plannedcommit_citycost     11493 non-null  int

In [4]:
#summary statistics of numerical columns
df.describe()

,magency,plannedcommit_ccnonexempt,plannedcommit_ccexempt,plannedcommit_citycost,plannedcommit_nccstate,plannedcommit_nccfederal,plannedcommit_nccother,plannedcommit_noncitycost,plannedcommit_total,adopt_ccnonexempt,...,commit_total,spent_ccnonexempt,spent_ccexempt,spent_citycost,spent_nccstate,spent_nccfederal,spent_nccother,spent_noncitycost,spent_total,spent_total_checkbooknyc
count,11493.000000,1.149300e+04,1.149300e+04,1.149300e+04,1.149300e+04,1.149300e+04,1.149300e+04,1.149300e+04,1.149300e+04,1.142600e+04,...,1.142600e+04,1.142600e+04,1.143300e+04,1.142600e+04,1.142600e+04,1.142600e+04,1.142600e+04,1.142600e+04,1.142600e+04,1.149300e+04
mean,654.230836,1.247033e+07,3.022238e+06,1.549257e+07,1.132850e+05,2.725248e+05,3.181650e+04,4.176263e+05,1.591019e+07,1.303613e+07,...,2.615465e+06,3.799140e+06,1.748703e+06,5.548910e+06,8.538574e+04,5.069455e+05,6.494804e+04,6.572793e+05,6.206190e+06,4.312708e+06
std,325.985181,1.159855e+08,4.560318e+07,1.246609e+08,3.728877e+06,4.941024e+06,1.047582e+06,6.574844e+06,1.252556e+08,1.165316e+08,...,3.468204e+07,3.523860e+07,3.814194e+07,5.239376e+07,2.381076e+06,1.053604e+07,1.305997e+06,1.112741e+07,5.514273e+07,3.216889e+07
min,35.000000,-2.790400e+07,-1.193000e+06,-2.790400e+07,-1.290000e+05,-3.363000e+06,0.000000e+00,-3.363000e+06,-2.790400e+07,-3.204944e+07,...,0.000000e+00,0.000000e+00,-2.309325e+04,-2.309325e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-2.309325e+04,0.000000e+00
25%,801.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.354350e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,841.000000,2.000000e+05,0.000000e+00,3.300000e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.900000e+05,2.530000e+05,...,0.000000e+00,4.523450e+02,0.000000e+00,4.779241e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.249960e+04,0.000000e+00
75%,846.000000,2.200000e+06,0.000000e+00,3.285000e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.528000e+06,2.402877e+06,...,2.155697e+05,1.031517e+06,0.000000e+00,1.398108e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.580759e+06,8.710552e+05
max,998.000000,4.406301e+09,3.430000e+09,4.406301e+09,3.479550e+08,3.850780e+08,9.850300e+07,3.850780e+08,4.458050e+09,4.024650e+09,...,1.952838e+09,2.058821e+09,2.781109e+09,2.848223e+09,2.175040e+08,6.388946e+08,9.470646e+07,6.388946e+08,2.848223e+09,1.464396e+09


In [5]:
#Checking for missing values
df.isnull().sum()

,0
ccpversion,0
maprojid,0
magencyacro,0
magency,0
magencyname,0
description,0
projectid,0
mindate,0
maxdate,0
typecategory,886


In [6]:
# Checking for duplicate rows that could bias the model
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate project IDs: {df['projectid'].duplicated().sum()}")

Duplicate rows: 0
Duplicate project IDs: 20


# **Data Pre-Processing & Cleaning**


In [7]:
# Budget columns contain $ and , symbols that need to be removed and converted to float


#Helper function to clean currency strings
def clean_currency(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, str):
        return float(x.replace('$', '').replace(',', '').strip())
    return float(x)

# Columns to convert
budget_columns = [col for col in df.columns
                  if any(x in col for x in ['plannedcommit', 'adopt', 'allocate', 'commit', 'spent'])]

for col in budget_columns:
    df[col] = df[col].apply(clean_currency)

display(df[budget_columns].head())

,plannedcommit_ccnonexempt,plannedcommit_ccexempt,plannedcommit_citycost,plannedcommit_nccstate,plannedcommit_nccfederal,plannedcommit_nccother,plannedcommit_noncitycost,plannedcommit_total,adopt_ccnonexempt,adopt_ccexempt,...,commit_total,spent_ccnonexempt,spent_ccexempt,spent_citycost,spent_nccstate,spent_nccfederal,spent_nccother,spent_noncitycost,spent_total,spent_total_checkbooknyc
0,9234000.0,29802000.0,39036000.0,0.0,0.0,0.0,0.0,39036000.0,9234000.0,29802000.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
1,100000.0,0.0,100000.0,0.0,0.0,0.0,0.0,100000.0,100000.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
3,393000.0,0.0,393000.0,0.0,0.0,0.0,0.0,393000.0,393400.0,0.0,...,6600.00,0.00,0.00,0.00,0.0,0.0,0.00,0.00,0.00,0.00
4,0.0,5787000.0,5787000.0,0.0,0.0,76000.0,76000.0,5863000.0,-98946.0,22368000.0,...,13133548.54,138249.54,10298251.65,10436501.19,0.0,0.0,1501431.87,1501431.87,11937933.06,6896227.51


In [8]:
#Handling missing values

# Drop rows with missing target variable as imputing would create fake ground truth values
# Fill missing typecategory with 'Unknown'

# Before
print(f"Rows before: {len(df)}")
print(f"Missing spent_total: {df['spent_total'].isna().sum()}")
print(f"Missing typecategory: {df['typecategory'].isna().sum()}")

# Drop rows where target is missing
df = df.dropna(subset=['spent_total', 'commit_total']).copy()

# Fill missing typecategory with 'Unknown'
df['typecategory'] = df['typecategory'].fillna('Unknown')

# After
print(f"Rows after dropping missing target: {len(df)}")
print(f"Missing typecategory after fill: {df['typecategory'].isna().sum()}")


Rows before: 11493
Missing spent_total: 67
Missing typecategory: 886
Rows after dropping missing target: 11426
Missing typecategory after fill: 0


In [9]:
#Feature engineering the target variable
#Target variable: budget_status Classify budget status based on variance percentage.
  # On Budget: Within 5% of committed funds
  # Over Budget: More than 5% over committed
  # Under Budget: More than 5% under committed

def classify_budget(row):
    if row['commit_total'] == 0:
        if row['spent_total'] == 0:
            return 'On Budget'
        else:
            return 'Over Budget'

    variance_pct = (row['spent_total'] - row['commit_total']) / abs(row['commit_total'])

    if variance_pct < -0.05:
        return 'Under Budget'
    elif variance_pct > 0.05:
        return 'Over Budget'
    else:
        return 'On Budget'

df['budget_status'] = df.apply(classify_budget, axis=1)

# Display class distribution
print("Class Distribution:")
class_counts = df['budget_status'].value_counts()
for status, count in class_counts.items():
    print(f"  {status:<15} {count:>6} ({count/len(df)*100:.1f}%)")

Class Distribution:
  Over Budget       5052 (44.2%)
  On Budget         4794 (42.0%)
  Under Budget      1580 (13.8%)


In [10]:
#Feature engineering project_duration_days from date columns

# Parse dates
df['mindate'] = pd.to_datetime(df['mindate'], format='%m/%d/%Y', errors='coerce')
df['maxdate'] = pd.to_datetime(df['maxdate'], format='%m/%d/%Y', errors='coerce')

# Create feature
df['project_duration_days'] = (df['maxdate'] - df['mindate']).dt.days

df['project_duration_days']

,project_duration_days
0,2435
1,0
2,1095
3,0
4,1905
...,...
11488,577
11489,638
11490,3752
11491,2842


In [11]:
#Encoding categorical variables

# Applying label encoding to typecategory as the number of categories is large
le_type = LabelEncoder()
df['typecategory_encoded'] = le_type.fit_transform(df['typecategory'])

# Applying label encoding to magencyname as the number of categories is large
le_agency = LabelEncoder()
df['magencyname_encoded'] = le_agency.fit_transform(df['magencyname'])

# Encoding target variable
le_target = LabelEncoder()
df['budget_status_encoded'] = le_target.fit_transform(df['budget_status'])

for i, cls in enumerate(le_target.classes_):
    print(f"  {i} = {cls}")

display(df[['typecategory_encoded', 'magencyname_encoded','budget_status_encoded']].head())

  0 = On Budget
  1 = Over Budget
  2 = Under Budget


,typecategory_encoded,magencyname_encoded,budget_status_encoded
0,3,8,0
1,0,4,0
2,3,21,0
3,3,21,2
4,2,8,2


In [12]:
#calculating the correlation for only numerical columns with the target variable spent_total in order to make feature selection

#Getting all numerical columns from the Dataset
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

#Removing the target variable itself from the list of features
if 'spent_total' in numeric_cols:
    numeric_cols.remove('budget_status_encoded')
#Calculating correlations
correlations = {}
for col in numeric_cols:
    if col in df.columns:
        corr = df[col].corr(df['budget_status_encoded'])
        correlations[col] = corr
# Create a DataFrame for correlations and sort by absolute value
corr_df = pd.DataFrame({
    'Feature': correlations.keys(),
    'Correlation': correlations.values()
})
corr_df['Abs_Correlation'] = corr_df['Correlation'].abs()
corr_df = corr_df.sort_values('Abs_Correlation', ascending=False).reset_index(drop=True)

print("Correlation with budget_status_encoded (sorted by strength):")
for _, row in corr_df.iterrows():
    strength = "STRONG" if row['Abs_Correlation'] > 0.5 else "MODERATE" if row['Abs_Correlation'] > 0.2 else "WEAK"
    print(f"  {row['Feature']:<30} {row['Correlation']:>8.4f}  ({strength})")

Correlation with budget_status_encoded (sorted by strength):
  project_duration_days            0.3705  (MODERATE)
  magency                          0.1506  (WEAK)
  typecategory_encoded            -0.1275  (WEAK)
  commit_total                     0.1035  (WEAK)
  allocate_citycost                0.0991  (WEAK)
  commit_citycost                  0.0991  (WEAK)
  commit_nccother                  0.0818  (WEAK)
  commit_noncitycost               0.0793  (WEAK)
  spent_total_checkbooknyc         0.0600  (WEAK)
  commit_ccexempt                  0.0591  (WEAK)
  commit_nccfederal                0.0532  (WEAK)
  spent_total                      0.0516  (WEAK)
  commit_ccnonexempt               0.0494  (WEAK)
  spent_ccnonexempt                0.0494  (WEAK)
  spent_citycost                   0.0491  (WEAK)
  adopt_ccexempt                   0.0340  (WEAK)
  adopt_nccfederal                 0.0325  (WEAK)
  magencyname_encoded             -0.0304  (WEAK)
  allocate_ccnonexempt            -

In [13]:
# FEATURE SELECTION ANALYSIS

# All potential features (excluding target-related columns)
potential_features = [
    'plannedcommit_total', 'adopt_total', 'allocate_total',
    'plannedcommit_citycost', 'adopt_citycost', 'allocate_citycost',
    'plannedcommit_nccstate', 'adopt_nccstate', 'allocate_nccstate',
    'plannedcommit_nccfederal', 'adopt_nccfederal', 'allocate_nccfederal',
    'project_duration_days',
    'typecategory_encoded', 'magencyname_encoded'
]

# Prepare data
df_temp = df[potential_features + ['budget_status_encoded']].dropna()
X_temp = df_temp[potential_features]
y_temp = df_temp['budget_status_encoded']

X_train_temp, X_test_temp, y_train_temp, y_test_temp = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

# Train Random Forest to get feature importance
rf_temp = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, n_jobs=-1)
rf_temp.fit(X_train_temp, y_train_temp)

# Get feature importance
importance_df = pd.DataFrame({
    'Feature': potential_features,
    'Importance': rf_temp.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance Analysis:")
print("-" * 50)
for _, row in importance_df.iterrows():
    status = "✓ KEEP" if row['Importance'] > 0.004 else "✗ DROP"
    print(f"  {row['Feature']:<28} {row['Importance']:.4f}  {status}")

# Select features with importance > 0.004
selected_features = importance_df[importance_df['Importance'] > 0.004]['Feature'].tolist()
print(f"\nSelected {len(selected_features)} features out of {len(potential_features)}")

Feature Importance Analysis:
--------------------------------------------------
  allocate_citycost            0.2754  ✓ KEEP
  project_duration_days        0.2332  ✓ KEEP
  adopt_total                  0.0997  ✓ KEEP
  adopt_citycost               0.0901  ✓ KEEP
  plannedcommit_citycost       0.0859  ✓ KEEP
  allocate_total               0.0840  ✓ KEEP
  plannedcommit_total          0.0742  ✓ KEEP
  magencyname_encoded          0.0286  ✓ KEEP
  typecategory_encoded         0.0099  ✓ KEEP
  adopt_nccfederal             0.0054  ✓ KEEP
  allocate_nccfederal          0.0046  ✓ KEEP
  plannedcommit_nccfederal     0.0032  ✗ DROP
  adopt_nccstate               0.0022  ✗ DROP
  allocate_nccstate            0.0021  ✗ DROP
  plannedcommit_nccstate       0.0016  ✗ DROP

Selected 11 features out of 15


In [14]:
#Feature selection

# Features selected based on Random Forest importance analysis
# Importance threshold > 0.004
# Excludes: state/federal funding (nccstate, nccfederal) - very low importance

feature_columns = [
    'allocate_citycost',        # 0.2690 - Highest importance
    'project_duration_days',    # 0.1408
    'adopt_total',              # 0.0642
    'plannedcommit_total',      # 0.0553
    'adopt_citycost',           # 0.0540
    'plannedcommit_citycost',   # 0.0527
    'allocate_total',           # 0.0470
    'magencyname_encoded',      # 0.0159
    'typecategory_encoded',     # 0.0048
]

target_column = 'budget_status_encoded'

# Creating final dataframe
df_model = df[feature_columns + [target_column]].copy()
df_model = df_model.dropna()

df_model.head()


,allocate_citycost,project_duration_days,adopt_total,plannedcommit_total,adopt_citycost,plannedcommit_citycost,allocate_total,magencyname_encoded,typecategory_encoded,budget_status_encoded
0,0.00,2435,39036000.0,39036000.0,39036000.0,39036000.0,39036000.0,8,3,0
1,0.00,0,100000.0,100000.0,100000.0,100000.0,100000.0,4,0,0
2,0.00,1095,0.0,0.0,0.0,0.0,0.0,21,3,0
3,6600.00,0,400000.0,393000.0,400000.0,393000.0,393400.0,21,3,2
4,9846362.06,1905,27656000.0,5863000.0,22864000.0,5787000.0,2584518.4,8,2,2


# **Modelling**

In [15]:
X = df_model[feature_columns]
y = df_model[target_column]

# Stratified split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
# logistic regression training
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [17]:
#Random forest training
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)


RandomForestClassifier(max_depth=20, min_samples_split=5, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [18]:
# XGBoost training
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=10,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=10, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...)

In [19]:
#model prediction
y_pred_lr = lr_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)

In [20]:
def evaluate_classifier(y_true, y_pred, model_name):

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted')
    rec = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')
    return {
        'Model': model_name,
        'Accuracy': f'{acc:.4f}',
        'Precision': f'{prec:.4f}',
        'Recall': f'{rec:.4f}',
        'F1-Score': f'{f1:.4f}'
    }

results = [
    evaluate_classifier(y_test, y_pred_lr, 'Logistic Regression'),
    evaluate_classifier(y_test, y_pred_rf, 'Random Forest'),
    evaluate_classifier(y_test, y_pred_xgb, 'XGBoost')
]

results_df = pd.DataFrame(results)
print( results_df.to_string(index=False))

              Model Accuracy Precision Recall F1-Score
Logistic Regression   0.8224    0.8510 0.8224   0.8196
      Random Forest   0.9313    0.9302 0.9313   0.9305
            XGBoost   0.9453    0.9450 0.9453   0.9451


In [21]:
cm = confusion_matrix(y_test, y_pred_rf)
print(f"Classes: {list(le_target.classes_)}")
print(cm)

Classes: ['On Budget', 'Over Budget', 'Under Budget']
[[940  13   6]
 [ 33 943  35]
 [  2  68 246]]


# Integrating XAI

In [22]:
#SHAP global explanation

# Create SHAP explainer
print("Initializing SHAP TreeExplainer...")
explainer = shap.TreeExplainer(rf_model)

# Calculate SHAP values
print("Calculating SHAP values")
X_sample = X_test.head(500)
shap_values = explainer.shap_values(X_sample)

# Process SHAP values
shap_array = np.array(shap_values)
print(f"SHAP values shape: {shap_array.shape}")

# Handle shape: could be (3, 500, 11) or (500, 11, 3)
if shap_array.shape[0] == 3 and shap_array.shape[0] != shap_array.shape[2]:
    # Shape is (3, 500, 11) - transpose to (500, 11, 3)
    shap_array = np.transpose(shap_array, (1, 2, 0))

# Calculate global feature importance (mean |SHAP| across all samples and classes)
global_importance = np.abs(shap_array).mean(axis=(0, 2))

shap_importance_df = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': global_importance
}).sort_values('Importance', ascending=False)

print()
for rank, (_, row) in enumerate(shap_importance_df.iterrows(), 1):
    pct = row['Importance'] / global_importance.sum() * 100
    bar = '█' * int(pct)
    print(f"  {rank}. {row['Feature']:<25} {row['Importance']:>10.2f} ({pct:>5.1f}%) {bar}")


Initializing SHAP TreeExplainer...
Calculating SHAP values
SHAP values shape: (500, 9, 3)

  1. allocate_citycost               0.15 ( 30.4%) ██████████████████████████████
  2. project_duration_days           0.12 ( 24.2%) ████████████████████████
  3. adopt_total                     0.05 ( 10.0%) █████████
  4. plannedcommit_citycost          0.04 (  8.9%) ████████
  5. plannedcommit_total             0.04 (  8.8%) ████████
  6. adopt_citycost                  0.04 (  8.7%) ████████
  7. allocate_total                  0.04 (  7.3%) ███████
  8. magencyname_encoded             0.01 (  1.3%) █
  9. typecategory_encoded            0.00 (  0.3%) 


In [23]:
#SHAP explanation per class

for class_idx, class_name in enumerate(le_target.classes_):
    print(f"\n--- {class_name} ---")

    # Get SHAP values for this class
    class_shap = np.abs(shap_array[:, :, class_idx]).mean(axis=0)

    class_importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': class_shap
    }).sort_values('Importance', ascending=False)

    print("Top 5 features:")
    for rank, (_, row) in enumerate(class_importance_df.head(5).iterrows(), 1):
        print(f"    {rank}. {row['Feature']:<25} {row['Importance']:.2f}")


--- On Budget ---
Top 5 features:
    1. allocate_citycost         0.20
    2. project_duration_days     0.13
    3. adopt_total               0.06
    4. allocate_total            0.05
    5. adopt_citycost            0.05

--- Over Budget ---
Top 5 features:
    1. project_duration_days     0.18
    2. allocate_citycost         0.08
    3. adopt_total               0.07
    4. plannedcommit_citycost    0.07
    5. plannedcommit_total       0.07

--- Under Budget ---
Top 5 features:
    1. allocate_citycost         0.18
    2. project_duration_days     0.05
    3. plannedcommit_total       0.02
    4. adopt_citycost            0.02
    5. plannedcommit_citycost    0.02


In [24]:
#LIME

# Create LIME explainer
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_columns,
    class_names=list(le_target.classes_),
    mode='classification',
    random_state=42
)


# Explain one example from each class
for target_class in range(3):
    # Find correctly classified examples
    correct_mask = (y_test.values == target_class) & (rf_model.predict(X_test) == target_class)
    indices = np.where(correct_mask)[0]

    if len(indices) > 0:
        idx = indices[0]
        instance = X_test.iloc[idx].values
        print(f"EXAMPLE PREDICTION: {le_target.classes_[target_class]}")

        # Show instance details
        print("Project Details:")
        for feat, val in zip(feature_columns, instance):
            print(f"    {feat}: {val:,.2f}")

        # Prediction probabilities
        proba = rf_model.predict_proba([instance])[0]
        print(f"Prediction Probabilities:")
        for i, cls in enumerate(le_target.classes_):
            bar = '█' * int(proba[i] * 30)
            print(f"    {cls:<15} {proba[i]:>7.1%} {bar}")

        # LIME explanation
        exp = lime_explainer.explain_instance(
            instance,
            rf_model.predict_proba,
            num_features=5,
            top_labels=1
        )



EXAMPLE PREDICTION: On Budget
Project Details:
    allocate_citycost: 0.00
    project_duration_days: 2,192.00
    adopt_total: 74,000,000.00
    plannedcommit_total: 74,000,000.00
    adopt_citycost: 74,000,000.00
    plannedcommit_citycost: 74,000,000.00
    allocate_total: 74,000,000.00
    magencyname_encoded: 15.00
    typecategory_encoded: 1.00
Prediction Probabilities:
    On Budget         91.5% ███████████████████████████
    Over Budget        8.5% ██
    Under Budget       0.0% 
EXAMPLE PREDICTION: Over Budget
Project Details:
    allocate_citycost: 0.00
    project_duration_days: 0.00
    adopt_total: 3,162,000.00
    plannedcommit_total: 0.00
    adopt_citycost: 3,162,000.00
    plannedcommit_citycost: 0.00
    allocate_total: 196,856.33
    magencyname_encoded: 12.00
    typecategory_encoded: 0.00
Prediction Probabilities:
    On Budget          3.7% █
    Over Budget       96.2% ████████████████████████████
    Under Budget       0.1% 
EXAMPLE PREDICTION: Under Budget
Pr

# Visualizations